# LayerLog Anatomy

Audits aggregate LayerLog fields, properties, and methods against a live layer record.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## LayerLog Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog",
    "tl.LayerLog.PORTABLE_STATE_SPEC",
    "tl.LayerLog.activation",
    "tl.LayerLog.activation_postfunc",
    "tl.LayerLog.activation_transform",
    "tl.LayerLog.autograd_saved_bytes",
    "tl.LayerLog.autograd_saved_tensor_count",
    "tl.LayerLog.buffer_address",
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.captured_args",
    "tl.LayerLog.captured_kwargs",
    "tl.LayerLog.child_layers",
    "tl.LayerLog.child_layers_per_pass",
    "tl.LayerLog.child_passes_per_layer",
    "tl.LayerLog.co_parent_layers",
    "tl.LayerLog.cond_branch_children_by_cond",
    "tl.LayerLog.cond_branch_elif_children",
    "tl.LayerLog.cond_branch_else_children",
    "tl.LayerLog.cond_branch_start_children",
    "tl.LayerLog.cond_branch_then_children",
    "tl.LayerLog.conditional_branch_stack_passes",
    "tl.LayerLog.conditional_branch_stacks",
    "tl.LayerLog.containing_module",
    "tl.LayerLog.containing_modules",
    "tl.LayerLog.corresponding_grad_fn",
    "tl.LayerLog.creation_order",
    "tl.LayerLog.detach_saved_tensor",
    "tl.LayerLog.edges_vary_across_passes",
    "tl.LayerLog.equivalent_operations",
    "tl.LayerLog.extra_data",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func_applied",
    "tl.LayerLog.func_argnames",
    "tl.LayerLog.func_call_stack",
    "tl.LayerLog.func_config",
    "tl.LayerLog.func_is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_time",
    "tl.LayerLog.get_child_layers",
    "tl.LayerLog.get_parent_layers",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn_object",
    "tl.LayerLog.gradient",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_gradient",
    "tl.LayerLog.has_parents",
    "tl.LayerLog.has_saved_activations",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer_layer",
    "tl.LayerLog.is_computed_inside_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input_layer",
    "tl.LayerLog.is_internally_initialized",
    "tl.LayerLog.is_internally_terminated",
    "tl.LayerLog.is_output_layer",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool_layer",
    "tl.LayerLog.iterable_output_index",
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.layer_total_num",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.layer_type_num",
    "tl.LayerLog.leaf_module_passes",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_nesting_depth",
    "tl.LayerLog.num_args",
    "tl.LayerLog.num_keyword_args",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params_total",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_passes",
    "tl.LayerLog.num_positional_args",
    "tl.LayerLog.operation_equivalence_type",
    "tl.LayerLog.operation_num",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.params_memory",
    "tl.LayerLog.params_memory_str",
    "tl.LayerLog.parent_layer_arg_locs",
    "tl.LayerLog.parent_layers",
    "tl.LayerLog.parent_layers_per_pass",
    "tl.LayerLog.parent_param_barcodes",
    "tl.LayerLog.parent_param_logs",
    "tl.LayerLog.parent_param_shapes",
    "tl.LayerLog.parent_passes_per_layer",
    "tl.LayerLog.pass_labels",
    "tl.LayerLog.pass_num",
    "tl.LayerLog.passes",
    "tl.LayerLog.save_gradients",
    "tl.LayerLog.scalar_bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.sibling_layers",
    "tl.LayerLog.source_trace",
    "tl.LayerLog.tensor",
    "tl.LayerLog.tensor_dtype",
    "tl.LayerLog.tensor_memory",
    "tl.LayerLog.tensor_memory_str",
    "tl.LayerLog.tensor_shape",
    "tl.LayerLog.transformed_activation",
    "tl.LayerLog.transformed_activation_dtype",
    "tl.LayerLog.transformed_activation_memory",
    "tl.LayerLog.transformed_activation_shape",
    "tl.LayerLog.transformed_gradient",
    "tl.LayerLog.transformed_gradient_dtype",
    "tl.LayerLog.transformed_gradient_memory",
    "tl.LayerLog.transformed_gradient_shape",
    "tl.LayerLog.uses_params",
]

pretty_print_fields(
    layer_log,
    [
        "layer_label",
        "layer_label_w_pass",
        "tensor_shape",
        "tensor_dtype",
        "has_parents",
        "has_children",
        "num_passes",
    ],
)
pass_records = (
    layer_log.passes.values() if hasattr(layer_log.passes, "values") else layer_log.passes
)
print("pass labels", [p.layer_label_w_pass for p in pass_records])

## Identity and tensor data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog",
    "tl.LayerLog.PORTABLE_STATE_SPEC",
    "tl.LayerLog.activation",
    "tl.LayerLog.activation_postfunc",
    "tl.LayerLog.activation_transform",
    "tl.LayerLog.autograd_saved_bytes",
    "tl.LayerLog.autograd_saved_tensor_count",
    "tl.LayerLog.buffer_address",
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.captured_args",
    "tl.LayerLog.captured_kwargs",
    "tl.LayerLog.child_layers",
    "tl.LayerLog.child_layers_per_pass",
    "tl.LayerLog.child_passes_per_layer",
    "tl.LayerLog.co_parent_layers",
    "tl.LayerLog.cond_branch_children_by_cond",
    "tl.LayerLog.cond_branch_elif_children",
    "tl.LayerLog.cond_branch_else_children",
    "tl.LayerLog.cond_branch_start_children",
    "tl.LayerLog.cond_branch_then_children",
    "tl.LayerLog.conditional_branch_stack_passes",
    "tl.LayerLog.conditional_branch_stacks",
    "tl.LayerLog.containing_module",
    "tl.LayerLog.containing_modules",
    "tl.LayerLog.corresponding_grad_fn",
]

audit_touch_items("Identity and tensor data", ITEMS, AUDIT_CONTEXT)

## Gradient and activation data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.creation_order",
    "tl.LayerLog.detach_saved_tensor",
    "tl.LayerLog.edges_vary_across_passes",
    "tl.LayerLog.equivalent_operations",
    "tl.LayerLog.extra_data",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func_applied",
    "tl.LayerLog.func_argnames",
    "tl.LayerLog.func_call_stack",
    "tl.LayerLog.func_config",
    "tl.LayerLog.func_is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_time",
    "tl.LayerLog.get_child_layers",
    "tl.LayerLog.get_parent_layers",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn_object",
    "tl.LayerLog.gradient",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_gradient",
    "tl.LayerLog.has_parents",
]

audit_touch_items("Gradient and activation data", ITEMS, AUDIT_CONTEXT)

## Graph relationships

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.has_saved_activations",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer_layer",
    "tl.LayerLog.is_computed_inside_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input_layer",
    "tl.LayerLog.is_internally_initialized",
    "tl.LayerLog.is_internally_terminated",
    "tl.LayerLog.is_output_layer",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool_layer",
    "tl.LayerLog.iterable_output_index",
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.layer_total_num",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.layer_type_num",
    "tl.LayerLog.leaf_module_passes",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
]

audit_touch_items("Graph relationships", ITEMS, AUDIT_CONTEXT)

## Module association

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_nesting_depth",
    "tl.LayerLog.num_args",
    "tl.LayerLog.num_keyword_args",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params_total",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_passes",
    "tl.LayerLog.num_positional_args",
    "tl.LayerLog.operation_equivalence_type",
    "tl.LayerLog.operation_num",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.params_memory",
    "tl.LayerLog.params_memory_str",
    "tl.LayerLog.parent_layer_arg_locs",
    "tl.LayerLog.parent_layers",
    "tl.LayerLog.parent_layers_per_pass",
    "tl.LayerLog.parent_param_barcodes",
    "tl.LayerLog.parent_param_logs",
    "tl.LayerLog.parent_param_shapes",
    "tl.LayerLog.parent_passes_per_layer",
    "tl.LayerLog.pass_labels",
    "tl.LayerLog.pass_num",
]

audit_touch_items("Module association", ITEMS, AUDIT_CONTEXT)

## Pass aggregation

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.LayerLog.passes",
    "tl.LayerLog.save_gradients",
    "tl.LayerLog.scalar_bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.sibling_layers",
    "tl.LayerLog.source_trace",
    "tl.LayerLog.tensor",
    "tl.LayerLog.tensor_dtype",
    "tl.LayerLog.tensor_memory",
    "tl.LayerLog.tensor_memory_str",
    "tl.LayerLog.tensor_shape",
    "tl.LayerLog.transformed_activation",
    "tl.LayerLog.transformed_activation_dtype",
    "tl.LayerLog.transformed_activation_memory",
    "tl.LayerLog.transformed_activation_shape",
    "tl.LayerLog.transformed_gradient",
    "tl.LayerLog.transformed_gradient_dtype",
    "tl.LayerLog.transformed_gradient_memory",
    "tl.LayerLog.transformed_gradient_shape",
    "tl.LayerLog.uses_params",
]

audit_touch_items("Pass aggregation", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog",
    "tl.LayerLog.PORTABLE_STATE_SPEC",
    "tl.LayerLog.activation",
    "tl.LayerLog.activation_postfunc",
    "tl.LayerLog.activation_transform",
    "tl.LayerLog.autograd_saved_bytes",
    "tl.LayerLog.autograd_saved_tensor_count",
    "tl.LayerLog.buffer_address",
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.captured_args",
    "tl.LayerLog.captured_kwargs",
    "tl.LayerLog.child_layers",
    "tl.LayerLog.child_layers_per_pass",
    "tl.LayerLog.child_passes_per_layer",
    "tl.LayerLog.co_parent_layers",
    "tl.LayerLog.cond_branch_children_by_cond",
    "tl.LayerLog.cond_branch_elif_children",
    "tl.LayerLog.cond_branch_else_children",
    "tl.LayerLog.cond_branch_start_children",
    "tl.LayerLog.cond_branch_then_children",
    "tl.LayerLog.conditional_branch_stack_passes",
    "tl.LayerLog.conditional_branch_stacks",
    "tl.LayerLog.containing_module",
    "tl.LayerLog.containing_modules",
    "tl.LayerLog.corresponding_grad_fn",
    "tl.LayerLog.creation_order",
    "tl.LayerLog.detach_saved_tensor",
    "tl.LayerLog.edges_vary_across_passes",
    "tl.LayerLog.equivalent_operations",
    "tl.LayerLog.extra_data",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func_applied",
    "tl.LayerLog.func_argnames",
    "tl.LayerLog.func_call_stack",
    "tl.LayerLog.func_config",
    "tl.LayerLog.func_is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_time",
    "tl.LayerLog.get_child_layers",
    "tl.LayerLog.get_parent_layers",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn_object",
    "tl.LayerLog.gradient",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_gradient",
    "tl.LayerLog.has_parents",
    "tl.LayerLog.has_saved_activations",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer_layer",
    "tl.LayerLog.is_computed_inside_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input_layer",
    "tl.LayerLog.is_internally_initialized",
    "tl.LayerLog.is_internally_terminated",
    "tl.LayerLog.is_output_layer",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool_layer",
    "tl.LayerLog.iterable_output_index",
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.layer_total_num",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.layer_type_num",
    "tl.LayerLog.leaf_module_passes",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_nesting_depth",
    "tl.LayerLog.num_args",
    "tl.LayerLog.num_keyword_args",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params_total",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_passes",
    "tl.LayerLog.num_positional_args",
    "tl.LayerLog.operation_equivalence_type",
    "tl.LayerLog.operation_num",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.params_memory",
    "tl.LayerLog.params_memory_str",
    "tl.LayerLog.parent_layer_arg_locs",
    "tl.LayerLog.parent_layers",
    "tl.LayerLog.parent_layers_per_pass",
    "tl.LayerLog.parent_param_barcodes",
    "tl.LayerLog.parent_param_logs",
    "tl.LayerLog.parent_param_shapes",
    "tl.LayerLog.parent_passes_per_layer",
    "tl.LayerLog.pass_labels",
    "tl.LayerLog.pass_num",
    "tl.LayerLog.passes",
    "tl.LayerLog.save_gradients",
    "tl.LayerLog.scalar_bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.sibling_layers",
    "tl.LayerLog.source_trace",
    "tl.LayerLog.tensor",
    "tl.LayerLog.tensor_dtype",
    "tl.LayerLog.tensor_memory",
    "tl.LayerLog.tensor_memory_str",
    "tl.LayerLog.tensor_shape",
    "tl.LayerLog.transformed_activation",
    "tl.LayerLog.transformed_activation_dtype",
    "tl.LayerLog.transformed_activation_memory",
    "tl.LayerLog.transformed_activation_shape",
    "tl.LayerLog.transformed_gradient",
    "tl.LayerLog.transformed_gradient_dtype",
    "tl.LayerLog.transformed_gradient_memory",
    "tl.LayerLog.transformed_gradient_shape",
    "tl.LayerLog.uses_params",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")